# Akash Setu — Phase 6: ML-Based Collision Risk Prediction
### End-to-End Conjunction Data Message (CDM) Risk Estimation Pipeline

---

## 1. Executive Summary & Scientific Context
Phase 6 introduces a machine learning-based collision risk prediction pipeline to the **Akash Setu** space situational awareness platform. 
The objective is to predict the final collision risk (measured as $\log_{10}(\text{Probability})$) of an orbital conjunction event from early Conjunction Data Messages (CDMs), enabling proactive operator decision-making before critical close approach (TCA).

### Astrodynamics & Machine Learning Foundations:
1. **Event-Level Splitting**: Conjunction events consist of sequential CDMs issued over several days. Splitting rows randomly across CDMs causes severe temporal data leakage. We enforce strict event-level partitioning where all CDMs belonging to a given `event_id` reside exclusively in the train or validation partition.
2. **Target Leakage Prevention**: Operational columns (`risk`, `true_risk`, `max_risk_estimate`, `max_risk_scaling`, `event_final_risk`, `event_id`, `mission_id`, `is_last_cdm`) are strictly purged from the feature matrix $X$.
3. **Censored Risk Floor (-30.0)**: Negligible collision probabilities in ESA CDMs are hard-censored at $\log_{10}(P) = -30.0$ (~77.5% of records). Metrics must evaluate overall performance as well as separate subgroups (clamped vs unclamped).
4. **Physical Interpretability**: Tree-based gradient boosted models capture non-linear conjunction geometries (Mahalanobis distance, covariance determinant volumes, relative velocity components) while maintaining rapid sub-millisecond inference latency.


In [1]:
# 2. Imports and Environment Setup
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    confusion_matrix
)
import joblib

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
print(f"Working directory root: {ROOT}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Working directory root: C:\Users\Pratham\Downloads\Programming Language\Projects\Aakash-Setu


## 2. Dataset Ingestion & Quality Audit
We ingest `data/processed/esa_train_features.parquet` (149,480 CDMs across 12,929 events) and `data/processed/esa_event_split.parquet` (pre-computed 80/20 train/validation partition).


In [2]:
FEAT_PATH = ROOT / 'data/processed/esa_train_features.parquet'
SPLIT_PATH = ROOT / 'data/processed/esa_event_split.parquet'

print(f"Loading features from: {FEAT_PATH.name}")
df_feat = pd.read_parquet(FEAT_PATH)
df_split = pd.read_parquet(SPLIT_PATH)

df = df_feat.merge(df_split, on='event_id', how='left')
print(f"Merged dataset shape: {df.shape}")
print(f"Total conjunction events: {df['event_id'].nunique():,}")
print("Split distribution:")
print(df['split'].value_counts())


Loading features from: esa_train_features.parquet
Merged dataset shape: (149480, 102)
Total conjunction events: 12,929
Split distribution:
split
train    120092
val       29388
Name: count, dtype: int64


## 3. Data Quality, Missing Values & Target Distribution Analysis
In ESA CDMs, the target `event_final_risk` is the log10 collision probability observed at the final available CDM for each event.


In [3]:
# Target distribution stats
y_all = df['event_final_risk']
floor_count = (y_all == -30.0).sum()
high_risk_count = (y_all >= -6.0).sum()

print("--- TARGET SUMMARY (event_final_risk) ---")
print(f"Mean:   {y_all.mean():.4f}")
print(f"Std:    {y_all.std():.4f}")
print(f"Min:    {y_all.min():.4f} (Censored floor at -30.0)")
print(f"Max:    {y_all.max():.4f}")
print(f"Clamped at -30.0: {floor_count:,} ({100 * floor_count / len(y_all):.2f}%)")
print(f"High-risk (>= -6.0): {high_risk_count:,} ({100 * high_risk_count / len(y_all):.2f}%)")

# Missing values check across features
NEVER_IN_X = ['risk', 'true_risk', 'max_risk_estimate', 'max_risk_scaling', 'event_final_risk', 'event_id', 'mission_id', 'is_last_cdm', 'split']
feature_cols = [c for c in df.columns if c not in NEVER_IN_X]
missing_counts = df[feature_cols].isnull().sum()
top_missing = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(f"\nTotal input features: {len(feature_cols)}")
print(f"Features with missing values: {len(top_missing)}")
print("Top 5 missing features:")
print(top_missing.head(5))


--- TARGET SUMMARY (event_final_risk) ---
Mean:   -26.1079
Std:    7.7967
Min:    -30.0000 (Censored floor at -30.0)
Max:    -1.6847
Clamped at -30.0: 115,864 (77.51%)
High-risk (>= -6.0): 2,126 (1.42%)

Total input features: 98
Features with missing values: 50
Top 5 missing features:
c_rcs_estimate    48360
c_crdot_r          7576
c_cndot_n          7576
c_cndot_rdot       7576
c_cndot_tdot       7576
dtype: int64


## 4. Preprocessing & Leakage-Free Partitions
We enforce strict event-level separation: all CDMs for a given `event_id` are placed into either the train partition or the validation partition. Preprocessing parameters (such as medians for imputation) are fitted strictly on `train_df`.


In [4]:
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()

# Verify zero event overlap
train_ev = set(train_df['event_id'].unique())
val_ev = set(val_df['event_id'].unique())
overlap = train_ev.intersection(val_ev)
assert len(overlap) == 0, f"Event leakage detected: {len(overlap)} events overlap!"
print(f"Train partition: {len(train_df):,} rows across {len(train_ev):,} events")
print(f"Val partition:   {len(val_df):,} rows across {len(val_ev):,} events")
print("Zero event leakage verified.")

from backend.ml_risk_engine import CDMEFeaturePreprocessor

preprocessor = CDMEFeaturePreprocessor().fit(train_df, feature_cols)
X_train = preprocessor.transform(train_df)
X_val = preprocessor.transform(val_df)
y_train = train_df['event_final_risk']
y_val = val_df['event_final_risk']
y_train_cls = (y_train >= -6.0).astype(int)
y_val_cls = (y_val >= -6.0).astype(int)


Train partition: 120,092 rows across 10,349 events
Val partition:   29,388 rows across 2,580 events
Zero event leakage verified.


## 5. Model Training & Baseline Benchmarks
We train and benchmark:
1. **Dummy Baseline**: Predicts the training set mean target value.
2. **Ridge Linear Baseline**: L2-regularized linear model with median imputation, standardization, and one-hot encoding.
3. **Primary Regressor (HistGradientBoostingRegressor)**: LightGBM-inspired tree booster that natively handles non-linearities and missing values.
4. **High-Risk Classifier (HistGradientBoostingClassifier)**: Binary classifier with balanced class weights to flag events crossing the ESA maneuver threshold ($\log_{10} P \ge -6.0$).


In [5]:
# Baseline 1: Dummy Mean Regressor
mean_train = float(y_train.mean())
y_pred_dummy = np.full(len(y_val), mean_train)
dummy_mae = mean_absolute_error(y_val, y_pred_dummy)
dummy_rmse = root_mean_squared_error(y_val, y_pred_dummy)

# Baseline 2: Ridge Linear Regression
num_cols = [c for c in feature_cols if c != 'c_object_type']
ridge_pipe = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sca', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['c_object_type'])
    ])),
    ('reg', Ridge(alpha=100.0, random_state=RANDOM_SEED))
])
print("Fitting Ridge baseline...")
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_val)
ridge_mae = mean_absolute_error(y_val, y_pred_ridge)
ridge_rmse = root_mean_squared_error(y_val, y_pred_ridge)
ridge_rho, _ = spearmanr(y_val, y_pred_ridge)

# Primary Model: HistGradientBoostingRegressor
print("Fitting HistGradientBoostingRegressor (Primary Model)...")
t0 = time.time()
regressor = HistGradientBoostingRegressor(
    max_iter=100,
    categorical_features=['c_object_type'],
    l2_regularization=0.1,
    min_samples_leaf=20,
    random_state=RANDOM_SEED
)
regressor.fit(X_train, y_train)
t_fit_reg = time.time() - t0
y_pred_reg = regressor.predict(X_val)
reg_mae = mean_absolute_error(y_val, y_pred_reg)
reg_rmse = root_mean_squared_error(y_val, y_pred_reg)
reg_rho, _ = spearmanr(y_val, y_pred_reg)
print(f"Primary Regressor fit in {t_fit_reg:.2f}s")

# High-Risk Classifier: HistGradientBoostingClassifier
print("Fitting HistGradientBoostingClassifier...")
clf = HistGradientBoostingClassifier(
    max_iter=100,
    class_weight='balanced',
    categorical_features=['c_object_type'],
    min_samples_leaf=20,
    random_state=RANDOM_SEED
)
clf.fit(X_train, y_train_cls)
y_prob_clf = clf.predict_proba(X_val)[:, 1]
clf_pr_auc = average_precision_score(y_val_cls, y_prob_clf)
clf_roc_auc = roc_auc_score(y_val_cls, y_prob_clf)
p, r, f1, _ = precision_recall_fscore_support(y_val_cls, (y_prob_clf >= 0.5).astype(int), average='binary')


Fitting Ridge baseline...


Fitting HistGradientBoostingRegressor (Primary Model)...


Primary Regressor fit in 2.63s
Fitting HistGradientBoostingClassifier...


## 6. Comprehensive Evaluation & Subgroup Metrics
We summarize overall validation performance and evaluate subgroup performance:
- **Clamped (-30.0)** vs **Unclamped (> -30.0)**
- **Lead-time subgroups**: $\le 2$ days, $2-5$ days, and $> 5$ days before TCA.


In [6]:
summary_data = [
    {'Model': 'Dummy Mean Baseline', 'MAE': f"{dummy_mae:.4f}", 'RMSE': f"{dummy_rmse:.4f}", 'Spearman Rho': 'N/A', 'PR-AUC': 'N/A'},
    {'Model': 'Ridge Linear Baseline', 'MAE': f"{ridge_mae:.4f}", 'RMSE': f"{ridge_rmse:.4f}", 'Spearman Rho': f"{ridge_rho:.4f}", 'PR-AUC': 'N/A'},
    {'Model': 'HistGradientBoostingRegressor (Ours)', 'MAE': f"{reg_mae:.4f}", 'RMSE': f"{reg_rmse:.4f}", 'Spearman Rho': f"{reg_rho:.4f}", 'PR-AUC': 'N/A'},
    {'Model': 'HistGradientBoostingClassifier', 'MAE': 'N/A', 'RMSE': 'N/A', 'Spearman Rho': 'N/A', 'PR-AUC': f"{clf_pr_auc:.4f}"}
]
summary_df = pd.DataFrame(summary_data)
print("=== MODEL COMPARISON SUMMARY ===")
print(summary_df.to_string(index=False))

# Subgroup breakdowns for primary regressor
clamped_mask = (y_val == -30.0)
unclamped_mask = ~clamped_mask

print("\n=== SUBGROUP BREAKDOWN (Censored Floor vs Unclamped) ===")
print(f"Clamped Floor (-30):   MAE={mean_absolute_error(y_val[clamped_mask], y_pred_reg[clamped_mask]):.4f}, RMSE={root_mean_squared_error(y_val[clamped_mask], y_pred_reg[clamped_mask]):.4f}")
print(f"Unclamped (>-30):     MAE={mean_absolute_error(y_val[unclamped_mask], y_pred_reg[unclamped_mask]):.4f}, RMSE={root_mean_squared_error(y_val[unclamped_mask], y_pred_reg[unclamped_mask]):.4f}, Spearman={spearmanr(y_val[unclamped_mask], y_pred_reg[unclamped_mask])[0]:.4f}")

# Time to TCA subgroups
t_val = X_val['time_to_tca'].to_numpy()
bins = {
    'Near Encounter (<= 2d)': t_val <= 2.0,
    'Mid-Range (2d - 5d)': (t_val > 2.0) & (t_val <= 5.0),
    'Early Warning (> 5d)': t_val > 5.0
}
print("\n=== LEAD TIME BREAKDOWN (Time to TCA) ===")
for b_name, b_mask in bins.items():
    b_mae = mean_absolute_error(y_val[b_mask], y_pred_reg[b_mask])
    b_rmse = root_mean_squared_error(y_val[b_mask], y_pred_reg[b_mask])
    b_rho = spearmanr(y_val[b_mask], y_pred_reg[b_mask])[0]
    print(f"{b_name:24s} | Samples: {b_mask.sum():5d} | MAE: {b_mae:.4f} | RMSE: {b_rmse:.4f} | Spearman: {b_rho:.4f}")


=== MODEL COMPARISON SUMMARY ===
                               Model    MAE   RMSE Spearman Rho PR-AUC
                 Dummy Mean Baseline 5.9990 7.6939          N/A    N/A
               Ridge Linear Baseline 5.0937 7.0256       0.3629    N/A
HistGradientBoostingRegressor (Ours) 3.8143 5.8967       0.5556    N/A
      HistGradientBoostingClassifier    N/A    N/A          N/A 0.1827

=== SUBGROUP BREAKDOWN (Censored Floor vs Unclamped) ===
Clamped Floor (-30):   MAE=2.4654, RMSE=3.9430
Unclamped (>-30):     MAE=8.4805, RMSE=10.0635, Spearman=0.4283

=== LEAD TIME BREAKDOWN (Time to TCA) ===
Near Encounter (<= 2d)   | Samples:  8066 | MAE: 2.6181 | RMSE: 4.6290 | Spearman: 0.5863
Mid-Range (2d - 5d)      | Samples: 13206 | MAE: 3.6944 | RMSE: 5.7571 | Spearman: 0.5373
Early Warning (> 5d)     | Samples:  8116 | MAE: 5.1981 | RMSE: 7.1189 | Spearman: 0.5168


## 7. Feature Importance & Physical Interpretability
We compute permutation feature importance on the validation set to evaluate which physical features drive the model's risk estimates.


In [7]:
val_sample = X_val.sample(n=3000, random_state=RANDOM_SEED)
val_y_sample = y_val.loc[val_sample.index]

res = permutation_importance(
    regressor,
    val_sample,
    val_y_sample,
    n_repeats=3,
    random_state=RANDOM_SEED,
    n_jobs=2
)
top_idx = res.importances_mean.argsort()[::-1][:15]
feat_names = val_sample.columns.tolist()

top_feat_df = pd.DataFrame({
    'Feature': [feat_names[i] for i in top_idx],
    'Importance Mean': [res.importances_mean[i] for i in top_idx],
    'Importance Std': [res.importances_std[i] for i in top_idx]
})
print("=== TOP 15 PHYSICAL DRIVERS OF COLLISION RISK ===")
print(top_feat_df.to_string(index=False))


=== TOP 15 PHYSICAL DRIVERS OF COLLISION RISK ===
                  Feature  Importance Mean  Importance Std
c_position_covariance_det         0.124086        0.006116
     mahalanobis_distance         0.095812        0.001571
            miss_distance         0.038177        0.004229
                c_sigma_t         0.036448        0.003145
              time_to_tca         0.029652        0.004192
      c_time_lastob_start         0.029247        0.004787
             c_sigma_rdot         0.024214        0.002074
      relative_velocity_t         0.018032        0.002262
                c_sigma_r         0.015988        0.001780
      relative_position_r         0.014891        0.001630
                t_j2k_inc         0.014179        0.002625
                   t_span         0.012663        0.002752
           t_rcs_estimate         0.012357        0.002770
             c_sigma_ndot         0.012174        0.002646
                  c_h_per         0.011776        0.001272


## 8. Artifact Persistence & Prediction API Verification
We persist the trained models and preprocessor to `backend/models/phase6/` and demonstrate inference using the production predictor.


In [8]:
MODEL_DIR = ROOT / 'backend/models/phase6'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(regressor, MODEL_DIR / 'risk_regressor.joblib')
joblib.dump(clf, MODEL_DIR / 'risk_classifier.joblib')
joblib.dump(preprocessor, MODEL_DIR / 'feature_preprocessor.joblib')

meta = {
    'model_version': '1.0.0-phase6',
    'training_timestamp': pd.Timestamp.utcnow().isoformat(),
    'n_train_samples': len(train_df),
    'n_val_samples': len(val_df),
    'n_train_events': int(train_df['event_id'].nunique()),
    'n_val_events': int(val_df['event_id'].nunique()),
    'n_features': len(feature_cols),
    'feature_names': feature_cols,
    'categorical_features': ['c_object_type'],
    'censored_risk_floor': -30.0,
    'action_threshold': -6.0,
    'top_features': [
        {'feature': feat_names[i], 'importance_mean': float(res.importances_mean[i]), 'importance_std': float(res.importances_std[i])}
        for i in top_idx
    ],
    'evaluation_summary': {
        'dummy_mean_val_rmse': float(dummy_rmse),
        'ridge_val_rmse': float(ridge_rmse),
        'primary_regressor_val_rmse': float(reg_rmse),
        'primary_regressor_val_spearman': float(reg_rho),
        'classifier_pr_auc': float(clf_pr_auc),
        'classifier_roc_auc': float(clf_roc_auc),
    },
}
with open(MODEL_DIR / 'model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Artifacts persisted successfully to:", MODEL_DIR)

from backend.ml_risk_engine import MLRiskPredictor
predictor = MLRiskPredictor.load(MODEL_DIR)

sample_cdm = {
    'time_to_tca': 1.25,
    'miss_distance': 450.0,
    'relative_speed': 14200.0,
    'c_object_type': 'DEBRIS',
    'mahalanobis_distance': 2.3,
    'c_position_covariance_det': 1.5e6
}
pred_result = predictor.predict_cdm(sample_cdm)
print("\nSample CDM Inference Result:")
for k, v in pred_result.items():
    print(f"  {k}: {v}")


Artifacts persisted successfully to: C:\Users\Pratham\Downloads\Programming Language\Projects\Aakash-Setu\backend\models\phase6

Sample CDM Inference Result:
  predicted_log10_risk: -21.2622
  clamped_log10_risk: -21.2622
  risk_probability: 5.4679e-22
  risk_category: LOW
  is_high_risk: True
  high_risk_probability: 0.6103
  action_recommendation: Low conjunction risk (log10 < -10.0). Routine monitoring.
  model_version: 1.0.0-phase6
  features_evaluated: 98
  scientific_caveat: Prototype ML model trained on historical ESA Kelvins competition CDMs. Outputs provide relative risk estimates and do not represent certified operational collision probabilities for active on-orbit assets.


C:\Users\Pratham\AppData\Local\Temp\ipykernel_40628\3979654986.py:10: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'training_timestamp': pd.Timestamp.utcnow().isoformat(),


## 9. Limitations & Operational Caveats
1. **Research Prototype**: Trained on historical, anonymized CDMs from the ESA Kelvins competition.
2. **Uncalibrated Real-World Probabilities**: Predictions indicate relative risk and conjunction severity; they do not replace rigorous full-covariance numerical SGP4/numerical screening.
3. **No ID-Joining**: Anonymized CDM objects must never be cross-joined with live CelesTrak NORAD catalog IDs by index or assumed keys.
